# Apply Rolling Trimmed Mean along the Raster Dimension
Need to reshape the FDM to create a raster dimension so that the rolling trimmed mean can be applied to each raster individually. Unfortunately, there are missing images in a few rasters, meaning that it is not trivial to reshape the array.
This Jupyter notebook performs the following steps to the output of the `despike_and_save.ipynb` notebook:
    - Identify the missing images and insert temporary nan images in those locations
    - Reshape the array to create the raster dimension
    - Apply the rolling trimmed mean along each raster
    - Reshape to the original array shape
    - Remove the temporary inserted images
    - Pickle and save the array
    
This array will then be used in the `background_subtraction.ipynb` Jupyter notebook.

#### Import Statements

In [ ]:
%reload_ext autoreload
%autoreload 2
# %matplotlib notebook
%matplotlib inline

import pathlib as pl
import numpy as np
import pickle
import numba
import matplotlib.pyplot as plt
# params = {"ytick.color" : "k",
#           "xtick.color" : "k",
#           "axes.labelcolor" : "k",
#           "axes.edgecolor" : "k"}
# plt.rcParams.update(params)
from mpl_toolkits.axes_grid1 import ImageGrid
# import pandas as pd
import ndfilters
from matplotlib import colors
import astropy.units as u
from astropy import constants as const
from iris_mosaics import wcs_to_bins, spectral_plot, read_sg_image, read_sg_image_lvl1
import iris_mosaics as iris_fdm
from IPython.display import display, Math, Markdown
from astropy.visualization import quantity_support, time_support
from scipy.optimize import curve_fit, minimize
from astropy.modeling import models, fitting
from astropy.convolution import convolve, Gaussian1DKernel, Gaussian2DKernel
import astropy.time
quantity_support()
time_support()

## Load data

#### File paths of Level 1.1 FDM data and FUV background image

In [ ]:
path = pl.Path(r'D:\IRIS data\deep_mosaics\20240811')

# Path of Level 1.1 spectrograph image files (used to save the files at the end using their original names but in a new folder)
path_fdm = path / 'level_11_plus_iris_prep_bg_sub'
files = list(path_fdm.glob('*.fits'))

# Paths of pickled despiked Level 1.1 data
path_dspk_1394 = path / 'level_11_fpr_despiked_1394.pickle'
path_dspk_1403 = path / 'level_11_fpr_despiked_1403.pickle'

#### Select Si IV 1394 & 1403 regions
Manually choose area where data is -- WCS doesn't seem to work -- my guess is because the geometric correction hasn't been applied yet.

In [ ]:
# Number of images
num_imgs = len(files)

# Read in an image in order to see where to crop it down:
file_num = 1000
w_0, hdu_0, _ = read_sg_image(files[file_num],'fuv2')
img_0 = hdu_0[0].data

# Lengths of dimensions of total image
num_y_total = img_0.shape[0]
num_x_total = img_0.shape[1]

# Create masks for the areas of interest (Si IV 1394 & 1403).
# sl_1394 = slice(10,~9), slice(732,~215)
# sl_1403 = slice(10,~9), slice(868,~7)

# For the Aug 2024 mosaic that is twice as wide in the x direction...
x1 = 2 * 732
x2 = 2 * 215
x3 = 2 * 868
x4 = 2 * 8
sl_1394 = slice(10,~9), slice(x1,~x2)
sl_1403 = slice(10,~9), slice(x3,~x4)

# Si IV 1394 and 1403 image dimension lengths
num_y = img_0[sl_1394].shape[0]
num_x_1394 = img_0[sl_1394].shape[1]
num_x_1403 = img_0[sl_1403].shape[1]

# Treat the upper and lower parts of the CCD separately
# There is an interface where the two detector taps sit next to each other, causing a discontinuity in the image.

# Define the 1394 upper and lower slices
sl_1394_up = slice(0,264)
sl_1394_down = slice(264, None)

# Define the 1403 upper and lower slices
sl_1403_up = slice(0,264)
sl_1403_down = slice(264, None)

# New y-axis shape
num_y_split = img_0[sl_1394_up].shape[0]

#### Read in despiked Level 1.1 data

In [ ]:
with open(str(path_dspk_1394), 'rb') as f:
    sg_1394_dspk = pickle.load(f)

with open(str(path_dspk_1403), 'rb') as f:
    sg_1403_dspk = pickle.load(f)
    
# Create NaN mask
nan_mask_1394 = ~np.isfinite(sg_1394_dspk)
nan_mask_1403 = ~np.isfinite(sg_1403_dspk)

# Mean images of the despiked data
sg_1394_dspk_mean_image = np.nanmean(sg_1394_dspk, axis=0)
sg_1403_dspk_mean_image = np.nanmean(sg_1403_dspk, axis=0)

#### Define plot function to better display Si IV 1394 and 1403 together

In [ ]:
def plot_lines_sidebyside(
        si_iv_1394, si_iv_1403, title,
        percentile_min: float = 0,
        percentile_max: float = 100,
        size: tuple = (5,5),
        exp_min = None,
        exp_max = None,
):
    # Set up figure and image grid
    fig = plt.figure(figsize=size)
    grid = ImageGrid(fig,
                     111,          # as in plt.subplot(111)
                     nrows_ncols=(1,2),
                     axes_pad=0.15,
                     # share_all=True,
                     cbar_location="right",
                     cbar_mode="single",
                     cbar_size="20%",
                     cbar_pad=0.15,
                     )

    if exp_min is None:
        exp_min = np.nanpercentile(si_iv_1394, percentile_min)
    if exp_max is None:
        exp_max = np.nanpercentile(si_iv_1394, percentile_max)

    im1 = grid[0].imshow(si_iv_1394, vmin=exp_min, vmax=exp_max)
    im2 = grid[1].imshow(si_iv_1403, vmin=exp_min, vmax=exp_max)
    # grid[0].set_title('1394 $\AA$', color='white')
    # grid[1].set_title('1403 $\AA$', color='white')
    grid[0].set_title('1394 Å')
    grid[1].set_title('1403 Å')
    # fig.suptitle(title, color='white')
    fig.suptitle(title)

    # Colorbar
    grid[~0].cax.colorbar(im2).set_label('DN', rotation=270)
    # grid[~0].cax.toggle_label(True)
    grid[0].invert_yaxis()
    # plt.tight_layout()    # Works, but may still require rect parameter to keep colorbar labels visible
    # plt.show()
    return fig

### Separate images into 64-step rasters
We will apply a rolling median filter to each raster separately since there are harsh breaks between each of them

#### Identify and fill in missing images

We need 64 images/raster before we can create a new raster dimension in the data.

**Method:**

Each raster scans in the positive x direction, so `diff(solar_x)` will always be positive within a raster (and will be around the known step size of 2 arcsec). Since rasters overlap, the jump from one raster to the next will always result in a negative `diff(solar_x)`, including the jump to a raster in a new row or the jump to a raster in a new column (see plot of image number vs solar x and solar y). We will find the missing images within a raster using a few different methods depending on the missing images' situations.

Images missing from within a raster (excluding the first and last images) are identified by the 4-arcsec difference in the `solar_x` array. We want to fill in those images working backwards through the array, otherwise the index of the missing images will change.

Once these are filled, we find images missing from either end of each raster by counting the images between each negative jump in `diff(solar_x)` (which indicates a jump from raster to raster) and see if any rasters have less than 64 images. The missing image(s) can be added to either end of the raster regardless of which was actually missing (not sure how to tell which was missing anyway). We again fill these in working backwards so the indices don't change as we add in missing images.

Define `solar_x` and `solar_y`

In [ ]:
%%time

solar_x = []
solar_y = []
t_obs = []

for i, file in enumerate(files):
    # Read in full image
    _, hdu, _ = read_sg_image(file,'fuv2')
    hx = hdu[0].header['CRVAL3']
    hy = hdu[0].header['CRVAL2']
    ht = hdu[0].header['T_OBS']
    solar_x.append(hx)
    solar_y.append(hy)
    t_obs.append(ht)

solar_x = np.array(solar_x)
solar_y = np.array(solar_y)
t_obs = astropy.time.Time(t_obs)

Plots of `diff(solar_x)` vs image # and `solar_x` and `solar_y` vs image #

In [ ]:
# Images/raster
num_img_per_raster = 64

# Ideally where the breaks between rasters should be
index_raster = np.arange(0, solar_x.shape[0], num_img_per_raster)

plt.figure(figsize=(9,7))
plt.plot(np.diff(solar_x))
# plt.ylim((-.5,4.5))
plt.ylim((-.5,35))
# plt.xlim((10225,10375))
for i in index_raster:
    plt.axvline(i, linewidth=0.5, color='grey')
plt.ylabel('diff(solar_x) [arcsec]')
plt.xlabel('image #')

plt.figure(figsize=(9,7))
# plt.scatter(solar_x[11456-64:11456+64], solar_y[11456-64:11456+64], c=np.arange(128))
plt.scatter(solar_x, solar_y, c=np.arange(solar_x.shape[0]),s=4)
# plt.scatter(solar_x[704:900], solar_y[704:900], c=np.arange(solar_x[704:900].shape[0]),s=0.5)
# i=866
# plt.scatter(solar_x[i], solar_y[i], c='r',s=1)
# plt.xlim((100,700))
# plt.ylim((700,800))
plt.colorbar(label='image #')
plt.xlabel('solar x [arcsec]')
plt.ylabel('solar y [arcsec]')

Identify where `diff(solar_x)` is larger than the approximate 2-arcsec step that is typical within a raster and flip array backwards so it fills in correctly

In [ ]:
missing_image_index = np.nonzero((np.diff(solar_x) > 3) & (np.diff(solar_x) < 5))[0][::-1]
missing_image_index

Insert NaNs for the missing images and fill in the `solar_x` array to indicate we filled in these images

In [ ]:
for i in missing_image_index:
    sg_1394_dspk = np.insert(sg_1394_dspk, i + 1, np.nan, axis=0)
    sg_1403_dspk = np.insert(sg_1403_dspk, i + 1, np.nan, axis=0)

# Add missing points in solar_x array by adding 2 arcsec to the previous point (2 arcsec steps within rasters)
solar_x_filled = solar_x.copy()
for i in missing_image_index:
    solar_x_filled = np.insert(solar_x_filled, i + 1, solar_x_filled[i] + 2)

Identify where `diff(solar_x)` goes negative -- this is where we jump from one raster to the next
Then insert 0 at the beginning and append the length of the array at the end so `diff(raster_jump_index)` will give the correct number of rasters in the next step

In [ ]:
raster_jump_index = np.nonzero(np.diff(solar_x_filled) < 0)[0] + 1
raster_jump_index = np.insert(raster_jump_index, 0, 0)
raster_jump_index = np.append(raster_jump_index, len(solar_x_filled))

Identify rasters containing less than 64 images and therefore missing an image(s) on the ends

In [ ]:
# Length of each raster
length_of_rasters = np.diff(raster_jump_index)

# Indices of rasters with less than 64 images, sorted backwards
raster_missing_end_index = np.nonzero(length_of_rasters < num_img_per_raster)[0][::-1]

# How many images are missing from the rasters with less than 64 images
num_missing_images = num_img_per_raster - length_of_rasters[raster_missing_end_index]

Fill in missing images with NaNs

In [ ]:
for i in range(len(raster_missing_end_index)):
    start_index = raster_jump_index[raster_missing_end_index[i]]
    nan_array =  np.array([np.nan] * num_missing_images[i])[..., np.newaxis, np.newaxis]
    sg_1394_dspk = np.insert(sg_1394_dspk, start_index, nan_array, axis=0)
    sg_1403_dspk = np.insert(sg_1403_dspk, start_index, nan_array, axis=0)

#### Reshape to create raster dimension

In [ ]:
# Reshape images dimension into a rasters dimension and an images/raster dimension
sg_1394_dspk_reshape = sg_1394_dspk.reshape(-1,num_img_per_raster,sg_1394_dspk.shape[~1], sg_1394_dspk.shape[~0])
sg_1403_dspk_reshape = sg_1403_dspk.reshape(-1,num_img_per_raster,sg_1403_dspk.shape[~1], sg_1403_dspk.shape[~0])

In [ ]:
plt.figure()
plt.imshow(np.nanmean(sg_1394_dspk_reshape[0], axis=(-1)).T, aspect=1/6)
plt.colorbar()
plt.gca().invert_yaxis()
plt.title('Raster 0 before rolling trimmed mean')

#### Apply rolling trimmed mean along raster dimension
Using 'reflect' at the boundaries.

In [ ]:
%%time
# ROY'S NEW TRIMMED MEAN FILTER

# Trimmed mean applied to individual Si IV 1394 rasters:

average_1394 = np.empty_like(sg_1394_dspk_reshape)
for i in range(average_1394.shape[0]):
    average_1394[i] = ndfilters.trimmed_mean_filter(
        array=sg_1394_dspk_reshape[i],
        size=(32,1,1),
        where=np.isfinite(sg_1394_dspk_reshape[i]),
        proportion=0.35
    )

del sg_1394_dspk_reshape

In [ ]:
plt.figure()
plt.imshow(np.nanmean(average_1394[0], axis=(-1)).T, aspect=1/6)
plt.colorbar()
plt.gca().invert_yaxis()
plt.title('Raster 0 after rolling trimmed mean')

In [ ]:
%%time
# ROY'S NEW TRIMMED MEAN FILTER

# Trimmed mean applied to individual Si IV 1403 rasters:

average_1403 = np.empty_like(sg_1403_dspk_reshape)
for i in range(average_1403.shape[0]):
    average_1403[i] = ndfilters.trimmed_mean_filter(
        array=sg_1403_dspk_reshape[i],
        size=(32, 1, 1),
        where=np.isfinite(sg_1403_dspk_reshape[i]),
        proportion=0.35
    )

del sg_1403_dspk_reshape

#### Reshape back into original shape (remove raster dimension)

In [ ]:
average_1394 = average_1394.reshape(-1,average_1394.shape[~1],average_1394.shape[~0])
average_1403 = average_1403.reshape(-1,average_1403.shape[~1],average_1403.shape[~0])

#### Remove inserted images from original data and average array

In [ ]:
missing_image_index_reverse = missing_image_index[::-1]

for i in missing_image_index_reverse:
    sg_1394_dspk = np.delete(sg_1394_dspk, [i+1], axis=0)
    sg_1403_dspk = np.delete(sg_1403_dspk, [i+1], axis=0)
    
    average_1394 = np.delete(average_1394, [i+1], axis=0)
    average_1403 = np.delete(average_1403, [i+1], axis=0)
    
raster_missing_end_index_reverse = raster_missing_end_index[::-1]

for i in range(len(raster_missing_end_index_reverse)):
    start_index = raster_jump_index[raster_missing_end_index_reverse[i]]
    
    sg_1394_dspk = np.delete(sg_1394_dspk, np.arange(start_index, start_index + num_missing_images[i]), axis=0)
    sg_1403_dspk = np.delete(sg_1403_dspk, np.arange(start_index, start_index + num_missing_images[i]), axis=0)
    
    average_1394 = np.delete(average_1394, np.arange(start_index, start_index + num_missing_images[i]), axis=0)
    average_1403 = np.delete(average_1403, np.arange(start_index, start_index + num_missing_images[i]), axis=0)

#### Reapply NaN masks

In [ ]:
average_1394[nan_mask_1394] = np.nan
average_1403[nan_mask_1403] = np.nan

#### Plots

In [ ]:
# For plotting purposes, take the mean of each of the trimmed mean arrays, so we have representative images for both.
average_1394_mean_image = np.nanmean(average_1394, axis=0)
average_1403_mean_image = np.nanmean(average_1403, axis=0)

In [ ]:
plot_lines_sidebyside(
    average_1394_mean_image,
    average_1403_mean_image,
    'Trimmed mean',
    size=(7,12),
    # percentile_min=.1,
    # percentile_max=99.9,
    # exp_max=35,
    # exp_min=3
);
 # .savefig('trimmed_mean_mar_2024.png',dpi=300, transparent=True)

#### Save rolling trimmed mean array

In [ ]:
with open(path / 'level_11_fpr_despiked_rtm_1394.pickle', 'wb') as fh:
    pickle.dump(average_1394, fh)

with open(path / 'level_11_fpr_despiked_rtm_1403.pickle', 'wb') as fh:
    pickle.dump(average_1403, fh)